# P1-E16.7H-2A Renamed (P1-E11.H_2A)Provenance-Prioritized Artifact Resolution Supplement
Purpose
Resolves the locked feature matrices and predictions needed for supplementary probability-quality and dataset-shift analyses.
Records the resolved sources and hashes so the results can be traced and regenerated.



## 1. Configuration and immutable contracts

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, log_loss, matthews_corrcoef, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED=20261012
TASKS={'normal_vs_ad':['Normal','AD'],'mci_vs_ad':['MCI','AD'],
       'normal_vs_mci':['Normal','MCI'],'three_class':['Normal','MCI','AD']}
TASK_LABELS={'normal_vs_ad':'Normal vs AD','mci_vs_ad':'MCI vs AD',
             'normal_vs_mci':'Normal vs MCI','three_class':'Three class'}
METHODS=['Frozen conventional benchmark','Endpoint-selected CEEDNet/SSL']
PROJECT_ROOT=Path(r'D:\EEG_AD_Research\dementia\paper one')
OUTPUTS_ROOT=PROJECT_ROOT/'paper1_outputs'
LEGACY_ROOT=Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
H2_ROOT=OUTPUTS_ROOT/'p1_e16_7h_2_external_compatibility_domain_shift_calibration_probability_quality'
G_ROOT=OUTPUTS_ROOT/'p1_e16_7g_four_endpoint_statistical_reproducibility_publication_lock'
BR_ROOT=OUTPUTS_ROOT/'p1_e16_7b_r_resume_safe_unified_four_endpoint_nested_method_comparison'
OUTPUT_ROOT=OUTPUTS_ROOT/'p1_e16_7h_2a_exact_artifact_resolution_supplement'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'
for d in (OUTPUT_ROOT,TABLES,FIGURES): d.mkdir(parents=True,exist_ok=True)

EXACT_DEVELOPMENT_OOF=None
EXACT_CAUEEG_FEATURES=None
EXACT_AHEPA_FEATURES=None
EXACT_DRYAD_FEATURES=None

def sha256_file(path,block=1<<20):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(block),b''): h.update(chunk)
    return h.hexdigest()
def save_table(q,name): q.to_csv(TABLES/f'{name}.csv',index=False)
def save_figure(fig,name):
    fig.tight_layout(); fig.savefig(FIGURES/f'{name}.pdf',bbox_inches='tight')
    fig.savefig(FIGURES/f'{name}.png',dpi=500,bbox_inches='tight',facecolor='white')
sns.set_theme(style='whitegrid',context='paper',font_scale=1.05)
plt.rcParams.update({'pdf.fonttype':42,'ps.fonttype':42,'savefig.dpi':500})
print({'output':str(OUTPUT_ROOT),'scope':'artifact resolution only; no disease-model modification'})

## 2. Discover paths recorded by upstream provenance

In [ ]:
def recursive_strings(value):
    if isinstance(value,dict):
        for v in value.values(): yield from recursive_strings(v)
    elif isinstance(value,(list,tuple)):
        for v in value: yield from recursive_strings(v)
    elif isinstance(value,str): yield value

def provenance_references(roots):
    rows=[]
    for root in roots:
        root=Path(root)
        if not root.is_dir(): continue
        for record in root.rglob('run_record.json'):
            try:
                for value in recursive_strings(json.loads(record.read_text(encoding='utf-8'))):
                    candidate=Path(value)
                    if candidate.is_file(): rows.append((candidate,record))
            except Exception: pass
        for manifest in root.rglob('*.csv'):
            if not any(x in manifest.name.lower() for x in ('manifest','provenance','source','artifact','input')): continue
            try:
                q=pd.read_csv(manifest,nrows=10000)
                for column in q:
                    if any(x in column.lower() for x in ('path','file','artifact')):
                        for value in q[column].dropna().astype(str):
                            candidate=Path(value)
                            if candidate.is_file(): rows.append((candidate,manifest))
            except Exception: pass
    return rows

references=provenance_references([G_ROOT,BR_ROOT,H2_ROOT,OUTPUTS_ROOT])
reference_table=pd.DataFrame([{'path':str(p),'source':str(s),'sha256':sha256_file(p)} for p,s in references]).drop_duplicates()
save_table(reference_table,'01_DISCOVERED_PROVENANCE_REFERENCES')
print('Recorded paths discovered:',len(reference_table))

## 3. Resolve the exact four-endpoint development OOF table

In [ ]:
def normalize_predictions(q):
    rename={}
    for old,new in [('endpoint','task'),('label','true_label'),('y_true','true_label'),
                    ('prediction','predicted_label'),('y_pred','predicted_label'),('model','method')]:
        if old in q and new not in q: rename[old]=new
    return q.rename(columns=rename)

def oof_contract(path):
    try:
        q=normalize_predictions(pd.read_csv(path))
        required={'task','method','true_label','predicted_label'}
        if not required.issubset(q): return False,'missing '+str(sorted(required-set(q)))
        if set(q.task.dropna())!=set(TASKS): return False,'endpoint set differs'
        unit=next((c for c in ('recording_id','participant_id') if c in q),None)
        if unit is None: return False,'analytical-unit ID absent'
        if q.duplicated(['task','method',unit]).any(): return False,'duplicate task-method-unit rows'
        for task,classes in TASKS.items():
            z=q.loc[q.task.eq(task)]
            if set(z.true_label.dropna())!=set(classes): return False,f'{task}: classes differ'
            if not set(f'probability_{c}' for c in classes).issubset(z): return False,f'{task}: probabilities absent'
        return True,f'rows={len(q)}; methods={q.method.nunique()}; unit={unit}'
    except Exception as exc: return False,type(exc).__name__+': '+str(exc)

def audit_oof(paths,tier):
    rows=[]
    for path in sorted({Path(p) for p in paths if Path(p).is_file()}):
        valid,detail=oof_contract(path)
        rows.append({'priority_tier':tier,'path':str(path),'sha256':sha256_file(path),
                     'contract_valid':valid,'detail':detail})
    return pd.DataFrame(rows)

tiers=[]
if EXACT_DEVELOPMENT_OOF:
    tiers.append(('manual exact override',[Path(EXACT_DEVELOPMENT_OOF)]))
else:
    # Tier 1: final tables from the exact P1-E16.7B-R experiment.
    tiers.append(('P1-E16.7B-R final tables',
      list((BR_ROOT/'tables').rglob('*PREDICTION*.csv')) if (BR_ROOT/'tables').is_dir() else []))
    # Tier 2: non-checkpoint files anywhere under the exact B-R root.
    br_files=[]
    if BR_ROOT.is_dir():
        br_files=[p for p in BR_ROOT.rglob('*PREDICTION*.csv')
                  if not any(x in str(p).lower() for x in ('checkpoint','partial','temporary','backup'))]
    tiers.append(('P1-E16.7B-R non-checkpoint outputs',br_files))
    # Tier 3: paths explicitly referenced by the final G provenance.
    if len(reference_table):
        gmask=reference_table.source.astype(str).str.lower().str.contains('p1_e16_7g')
        gpaths=[Path(p) for p in reference_table.loc[gmask,'path']
                if 'predict' in Path(p).name.lower()]
    else: gpaths=[]
    tiers.append(('P1-E16.7G referenced paths',gpaths))
    # Tier 4 is diagnostic only and is used only if final-chain tiers contain no valid table.
    broad=[Path(p) for p in reference_table.path if 'predict' in Path(p).name.lower()] if len(reference_table) else []
    tiers.append(('broader recorded provenance fallback',broad))

audits=[]; selected=None
for tier,paths in tiers:
    q=audit_oof(paths,tier); audits.append(q)
    valid=q.loc[q.contract_valid] if len(q) else pd.DataFrame()
    if len(valid) and valid.sha256.nunique()==1:
        selected=Path(valid.iloc[0].path); selected_tier=tier; break
    if len(valid) and valid.sha256.nunique()>1:
        # Within the final tables, prefer a single explicitly OOF-named content family.
        narrowed=valid.loc[valid.path.astype(str).str.lower().str.contains('oof')]
        if len(narrowed) and narrowed.sha256.nunique()==1:
            selected=Path(narrowed.iloc[0].path); selected_tier=tier+' / explicit OOF'; break
        display(valid[['priority_tier','path','sha256','detail']].to_string(index=False))
        raise RuntimeError(f'{tier}: several different contract-valid OOF families remain. Set EXACT_DEVELOPMENT_OOF to one displayed full path.')
oof_inventory=pd.concat(audits,ignore_index=True) if audits else pd.DataFrame()
save_table(oof_inventory,'02_DEVELOPMENT_OOF_CANDIDATE_AUDIT')
if selected is None:
    valid=oof_inventory.loc[oof_inventory.contract_valid] if len(oof_inventory) else pd.DataFrame()
    display(valid[['priority_tier','path','sha256','detail']].to_string(index=False) if len(valid) else oof_inventory)
    raise FileNotFoundError('No exact final-chain four-endpoint OOF table passed. Set EXACT_DEVELOPMENT_OOF.')
development_path=selected
development=normalize_predictions(pd.read_csv(development_path))
print('Resolved tier:',selected_tier)
print('Resolved OOF:',development_path)
print('SHA-256:',sha256_file(development_path))

## 4. Resolve and authenticate the three compatible 646-feature matrices

In [ ]:
def feature_contract(path):
    try:
        with np.load(path,allow_pickle=False) as z:
            if not {'X','feature_names'}.issubset(z.files): return False,'missing X/feature_names'
            X=np.asarray(z['X']); names=z['feature_names'].astype(str)
        valid=X.ndim==2 and X.shape[1]==len(names)==646 and np.isfinite(X).all()
        return valid,f'shape={X.shape}'
    except Exception as exc: return False,type(exc).__name__+': '+str(exc)

# First choices are the exact matrices used by the conventional external pipeline.
locked_caueeg=LEGACY_ROOT/'p1_e08_1_leakage_free_nonspiking_benchmark'/'features'/'02_LOCKED_RECORDING_LEVEL_NONSPIKING_FEATURE_MATRIX.npz'
locked_ahepa=LEGACY_ROOT/'p1_e10_2_ahepa_frozen_normal_ad_external_validation'/'AHEPA_LOCKED_NONSPIKING_FEATURE_MATRIX.npz'
locked_dryad=LEGACY_ROOT/'p1_e10_3_dryad_frozen_three_endpoint_external_validation'/'DRYAD_LOCKED_NONSPIKING_FEATURE_MATRIX.npz'
specs={
 'CAUEEG':(EXACT_CAUEEG_FEATURES,locked_caueeg,[OUTPUTS_ROOT,LEGACY_ROOT],['*LOCKED*RECORDING*FEATURE*MATRIX*.npz','*LOCKED*NONSPIKING*FEATURE*MATRIX*.npz']),
 'AHEPA':(EXACT_AHEPA_FEATURES,locked_ahepa,[locked_ahepa.parent],['AHEPA_LOCKED_NONSPIKING_FEATURE_MATRIX.npz']),
 'Dryad':(EXACT_DRYAD_FEATURES,locked_dryad,[locked_dryad.parent],['DRYAD_LOCKED_NONSPIKING_FEATURE_MATRIX.npz'])}
feature_paths={}; audit=[]
for dataset,(exact,locked,roots,patterns) in specs.items():
    tiers=[('manual exact override',[Path(exact)])] if exact else [('locked external-pipeline path',[locked])]
    # Paths referenced by final B-R/G records are the second automatic tier.
    if not exact and len(reference_table):
        mask=reference_table.source.astype(str).str.lower().str.contains('p1_e16_7b|p1_e16_7g')
        refs=[Path(p) for p in reference_table.loc[mask,'path'] if Path(p).suffix.lower()=='.npz']
        tiers.append(('final-chain referenced NPZ',refs))
    discovered=[]
    if not exact:
        for root in roots:
            if Path(root).is_dir():
                for pattern in patterns: discovered.extend(Path(root).rglob(pattern))
        tiers.append(('bounded discovery fallback',discovered))
    chosen=None
    for tier,paths in tiers:
        rows=[]; accepted=[]
        for path in sorted({Path(p) for p in paths if Path(p).is_file()}):
            valid,detail=feature_contract(path); lower=path.name.lower()
            belongs=(dataset.lower() in lower) if dataset!='CAUEEG' else ('ahepa' not in lower and 'dryad' not in lower)
            okay=valid and belongs
            rows.append({'dataset_role':dataset,'priority_tier':tier,'path':str(path),
                         'sha256':sha256_file(path),'contract_valid':okay,'detail':detail})
            if okay: accepted.append(path)
        audit.extend(rows); hashes={sha256_file(p) for p in accepted}
        if len(hashes)==1:
            chosen=sorted(accepted,key=lambda p:(len(str(p)),str(p)))[0]; break
        if len(hashes)>1:
            display(pd.DataFrame(rows).loc[lambda x:x.contract_valid].to_string(index=False))
            raise RuntimeError(f'{dataset}/{tier}: several valid feature families remain. Set the corresponding EXACT_* path.')
    if chosen is None: raise FileNotFoundError(f'{dataset}: no compatible locked 646-feature matrix was resolved.')
    feature_paths[dataset]=chosen
feature_audit=pd.DataFrame(audit); save_table(feature_audit,'03_FEATURE_MATRIX_CANDIDATE_AUDIT')
display(feature_audit.loc[feature_audit.contract_valid])
feature_data={}
for dataset,path in feature_paths.items():
    with np.load(path,allow_pickle=False) as z: feature_data[dataset]=(np.asarray(z['X'],float),z['feature_names'].astype(str))
names=feature_data['CAUEEG'][1]
for dataset,(X,n) in feature_data.items():
    if not np.array_equal(n,names): raise AssertionError(f'{dataset}: 646-feature names/order differ from CAUEEG.')
print({k:str(v) for k,v in feature_paths.items()})

## 5. Internal–external probability-quality comparison

In [ ]:
external_metrics_path=H2_ROOT/'tables'/'03_EXTERNAL_DISCRIMINATION_AND_PROBABILITY_QUALITY.csv'
external_predictions_path=OUTPUTS_ROOT/'p1_e16_7e_frozen_external_reevaluation_transportability_shift_chronology'/'tables'/'07_MATCHED_FROZEN_EXTERNAL_PARTICIPANT_PREDICTIONS.csv'
if not external_metrics_path.is_file() or not external_predictions_path.is_file():
    raise FileNotFoundError('Completed P1-E16.7H-2 and P1-E16.7E outputs are required.')
external_metrics=pd.read_csv(external_metrics_path)

def quality(q,task):
    classes=TASKS[task]; c2i={c:i for i,c in enumerate(classes)}
    y=q.true_label.map(c2i).to_numpy(int); yp=q.predicted_label.map(c2i).to_numpy(int)
    P=q[[f'probability_{c}' for c in classes]].to_numpy(float)
    P=np.clip(P,1e-15,1); P=P/P.sum(1,keepdims=True); onehot=np.eye(len(classes))[y]
    prevalence=np.bincount(y,minlength=len(classes))/len(y)
    brier=float(np.mean(np.sum((P-onehot)**2,axis=1)))
    baseline=float(np.mean(np.sum((prevalence[None,:]-onehot)**2,axis=1)))
    return {'n':len(q),'balanced_accuracy':balanced_accuracy_score(y,yp),
            'macro_f1':f1_score(y,yp,average='macro',zero_division=0),'mcc':matthews_corrcoef(y,yp),
            'roc_auc':roc_auc_score(y,P[:,1]) if len(classes)==2 else roc_auc_score(y,P,multi_class='ovr',average='macro',labels=np.arange(len(classes))),
            'log_loss':log_loss(y,P,labels=np.arange(len(classes))),'multiclass_brier':brier,
            'prevalence_baseline_brier':baseline,'brier_skill_score':1-brier/baseline}
internal=[]
for (task,method),q in development.groupby(['task','method']):
    internal.append({'dataset':'CAUEEG development OOF','task':task,'method':method,**quality(q,task)})
internal=pd.DataFrame(internal)
combined=pd.concat([internal,external_metrics],ignore_index=True,sort=False)
save_table(internal,'04_DEVELOPMENT_OOF_PROBABILITY_QUALITY')
save_table(combined,'05_INTERNAL_EXTERNAL_PROBABILITY_QUALITY_COMPARISON')
display(combined[['dataset','task','method','n','balanced_accuracy','roc_auc','multiclass_brier','prevalence_baseline_brier','brier_skill_score']].round(4))

## 6. Feature shift and domain separability

In [ ]:
Xdev=feature_data['CAUEEG'][0]; shift=[]; domain=[]
for dataset in ('AHEPA','Dryad'):
    Xext=feature_data[dataset][0]
    pooled=np.sqrt((Xdev.var(0,ddof=1)+Xext.var(0,ddof=1))/2)
    smd=np.divide(Xext.mean(0)-Xdev.mean(0),pooled,out=np.zeros_like(pooled),where=pooled>1e-12)
    shift.extend({'dataset':dataset,'feature_name':name,'smd_external_minus_caueeg':v,'absolute_smd':abs(v)} for name,v in zip(names,smd))
    X=np.vstack([Xdev,Xext]); y=np.r_[np.zeros(len(Xdev),int),np.ones(len(Xext),int)]
    folds=min(5,int(np.bincount(y).min())); cv=StratifiedKFold(folds,shuffle=True,random_state=SEED)
    model=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),
                    ('classifier',LogisticRegression(max_iter=3000,class_weight='balanced',random_state=SEED))])
    p=cross_val_predict(model,X,y,cv=cv,method='predict_proba',n_jobs=-1)[:,1]
    domain.append({'external_dataset':dataset,'n_caueeg':len(Xdev),'n_external':len(Xext),
                   'cv_folds':folds,'domain_classifier_roc_auc':roc_auc_score(y,p)})
shift=pd.DataFrame(shift); domain=pd.DataFrame(domain)
save_table(shift,'06_FEATURE_SHIFT_STANDARDIZED_MEAN_DIFFERENCES')
save_table(domain,'07_DOMAIN_CLASSIFIER_SEPARABILITY')
display(shift.groupby('dataset').absolute_smd.agg(['median','mean','max']).round(4))
display(domain.round(4))

## 7. Publication-ready figures

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
sns.histplot(data=shift,x='absolute_smd',hue='dataset',element='step',bins=35,common_norm=False,ax=axes[0])
axes[0].axvline(.5,color='#B22222',ls='--'); axes[0].set_title('Feature shift relative to CAUEEG'); axes[0].set_xlabel('Absolute standardized mean difference')
sns.barplot(data=domain,x='external_dataset',y='domain_classifier_roc_auc',color='#4C78A8',ax=axes[1])
axes[1].axhline(.5,color='0.3',ls='--'); axes[1].set_ylim(.45,1); axes[1].set_title('CAUEEG–external dataset separability'); axes[1].set_ylabel('Cross-validated domain ROC-AUC')
save_figure(fig,'Figure_1_Feature_Shift_and_Domain_Separability'); plt.show()

plot=combined.loc[combined.method.isin(METHODS)].copy()
plot['label']=plot.apply(lambda r:f"{r.dataset} · {TASK_LABELS[r.task]} · {'Conv.' if r.method==METHODS[0] else 'Deep'}",axis=1)
fig,ax=plt.subplots(figsize=(10,7)); colors=['#4C78A8' if m==METHODS[0] else '#F28E2B' for m in plot.method]
ax.barh(np.arange(len(plot)),plot.brier_skill_score,color=colors); ax.axvline(0,color='0.25',ls='--')
ax.set_yticks(np.arange(len(plot))); ax.set_yticklabels(plot.label); ax.set_xlabel('Brier skill score versus prevalence reference'); ax.set_title('Internal and external probability quality')
save_figure(fig,'Figure_2_Internal_External_Brier_Skill'); plt.show()

## 8. Final provenance and integrity lock

In [ ]:
sources=[{'role':'development OOF predictions','path':str(development_path),'sha256':sha256_file(development_path)},
 {'role':'completed H2 external metrics','path':str(external_metrics_path),'sha256':sha256_file(external_metrics_path)},
 {'role':'completed E16.7E external predictions','path':str(external_predictions_path),'sha256':sha256_file(external_predictions_path)}]
for dataset,path in feature_paths.items(): sources.append({'role':f'{dataset} feature matrix','path':str(path),'sha256':sha256_file(path)})
save_table(pd.DataFrame(sources),'08_FINAL_INPUT_PROVENANCE')
checks=pd.DataFrame([
 {'check':'one exact OOF content family','pass':oof_inventory.loc[oof_inventory.contract_valid,'sha256'].nunique()==1},
 {'check':'four development endpoints present','pass':set(development.task)==set(TASKS)},
 {'check':'one exact feature family per dataset','pass':set(feature_paths)=={'CAUEEG','AHEPA','Dryad'}},
 {'check':'identical ordered 646-feature schema','pass':all(len(n)==646 and np.array_equal(n,names) for X,n in feature_data.values())},
 {'check':'all feature values finite','pass':all(np.isfinite(X).all() for X,n in feature_data.values())},
 {'check':'internal estimates use OOF predictions','pass':len(internal)>0},
 {'check':'external metrics consumed unchanged','pass':len(external_metrics)>0},
 {'check':'domain classifier evaluated by cross-validation','pass':len(domain)==2 and domain.cv_folds.ge(2).all()},
 {'check':'no disease-model training or external modification','pass':True},
 {'check':'no performance target used as gate','pass':True}])
save_table(checks,'09_FINAL_INTEGRITY_CHECKS'); display(checks)
if not checks['pass'].all(): raise AssertionError('P1-E16.7H-2A integrity failure.')
run={'experiment':'P1-E16.7H-2A exact artifact resolution supplement','status':'PASSED',
     'completed_utc':datetime.now(timezone.utc).isoformat(),'development_oof':str(development_path),
     'feature_paths':{k:str(v) for k,v in feature_paths.items()},'disease_model_retraining':False,
     'external_modification':False,'integrity_passed':True,'environment':{'python':sys.version,'platform':platform.platform()}}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run,indent=2),encoding='utf-8')
manifest=[]
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file() and path.name!='10_OUTPUT_MANIFEST.csv':
        manifest.append({'relative_path':str(path.relative_to(OUTPUT_ROOT)),'bytes':path.stat().st_size,'sha256':sha256_file(path)})
save_table(pd.DataFrame(manifest),'10_OUTPUT_MANIFEST')
print('P1-E16.7H-2A PASSED — EXACT ARTIFACT RESOLUTION SUPPLEMENT LOCKED')

## Interpretation guardrails

- Internal estimates are development out-of-fold evidence, not official-test or external evidence.
- External results are consumed unchanged from P1-E16.7H-2.
- Domain ROC-AUC measures dataset separability, not disease-classification performance or the cause of shift.
- AHEPA and Dryad remain separate; no external result may be used for tuning.